In [ ]:
# ── folder structure expected ──────────────────────────────────────────────
# put this notebook in the same folder as 'split/'
#
# your_folder/
# ├── train_classification.ipynb   ← this file
# └── split/
#     ├── train/
#     │   ├── glioma/
#     │   ├── meningioma/
#     │   ├── notumor/
#     │   └── pituitary/
#     ├── val/
#     └── test/

import numpy as np
import tensorflow as tf
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

split_path = Path('./split')
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

In [ ]:
# ── data pipeline ──────────────────────────────────────────────────────────

train_ds = tf.keras.utils.image_dataset_from_directory(
    split_path / 'train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    split_path / 'val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    split_path / 'test', image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

class_names = train_ds.class_names
print('Classes:', class_names)

preprocess = tf.keras.applications.inception_v3.preprocess_input
train_ds = train_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(lambda x, y: (preprocess(x), y)).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# ── model architecture ─────────────────────────────────────────────────────

NUM_CLASSES = len(class_names)

base_model = tf.keras.applications.InceptionV3(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(256, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ── feature extraction training ────────────────────────────────────────────

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'feature_extraction_best.keras', monitor='val_loss', save_best_only=True
)

history_fe = model.fit(
    train_ds, validation_data=val_ds,
    epochs=30, callbacks=[early_stop, checkpoint]
)

In [ ]:
# ── evaluate feature extraction ────────────────────────────────────────────

model = tf.keras.models.load_model('feature_extraction_best.keras')

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names)
plt.tight_layout()
plt.savefig('confusion_matrix_fe.png')
plt.close()

results_fe = {
    'run_name'    : 'feature_extraction',
    'params'      : {'learning_rate': 1e-3, 'batch_size': BATCH_SIZE, 'epochs': 30,
                     'base_model': 'InceptionV3', 'frozen': True},
    'history'     : history_fe.history,
    'test_metrics': {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall'   : recall_score(y_true, y_pred, average='weighted'),
        'f1'       : f1_score(y_true, y_pred, average='weighted')
    }
}

with open('results_fe.json', 'w') as f:
    json.dump(results_fe, f, indent=2)

print('saved → results_fe.json + confusion_matrix_fe.png + feature_extraction_best.keras')

In [ ]:
# ── fine-tuning ────────────────────────────────────────────────────────────

for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_ft = tf.keras.callbacks.ModelCheckpoint(
    'fine_tuning_best.keras', monitor='val_loss', save_best_only=True
)

history_ft = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=[early_stop, checkpoint_ft]
)

In [ ]:
# ── evaluate fine-tuning ───────────────────────────────────────────────────

model = tf.keras.models.load_model('fine_tuning_best.keras')

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names)
plt.tight_layout()
plt.savefig('confusion_matrix_ft.png')
plt.close()

results_ft = {
    'run_name'    : 'fine_tuning',
    'params'      : {'learning_rate': 1e-5, 'batch_size': BATCH_SIZE, 'epochs': 15,
                     'base_model': 'InceptionV3', 'frozen': False},
    'history'     : history_ft.history,
    'test_metrics': {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall'   : recall_score(y_true, y_pred, average='weighted'),
        'f1'       : f1_score(y_true, y_pred, average='weighted')
    }
}

with open('results_ft.json', 'w') as f:
    json.dump(results_ft, f, indent=2)

print('saved → results_ft.json + confusion_matrix_ft.png + fine_tuning_best.keras')